# drone-detect : réentraînement sur Colab ou Kaggle

Ce notebook refait toute la chaîne du dépôt sur un GPU en ligne, pour qui n'a pas de GPU local :

1. téléchargement des données ;
2. conversion au format YOLO et découpage train / val / test **par séquence** ;
3. entraînement de YOLO11n, avec les mêmes arguments que les runs locaux : R1 à 640 px, puis R2,
   un fine-tuning à 1280 px qui part des poids de R1 ;
4. évaluation sur le test ;
5. export ONNX ;
6. téléchargement des poids et des métriques.

Il n'appelle que les scripts du dépôt : voir le `README.md` et `docs/DATA.md` pour les détails.

**Avant de commencer**

- **Colab** : *Exécution → Modifier le type d'exécution → GPU* (un T4 suffit).
  **Kaggle** : *Settings → Accelerator → GPU* et *Internet on*.
- **Disque** : environ 5,3 Go à télécharger et 6 Go occupés à la fin (davantage pendant l'extraction).
- **Durée** : la préparation des données prend de l'ordre de 20 à 30 min (non mesuré sur Colab).
  L'entraînement est borné par `--time` : 3 h pour R1, 2 h pour R2. Une session Colab gratuite
  peut s'arrêter avant : la section 4 sauvegarde les runs sur Google Drive pour pouvoir reprendre.
- **Licences des données** : Halmstad (CC0-1.0) et DUT Anti-UAV (Apache-2.0). Elles sont téléchargées
  depuis leurs sources et ne doivent pas être redistribuées sans respecter ces licences.

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi

## 2. Cloner le dépôt

Remplacer `<pseudo>` par le compte GitHub qui héberge le dépôt.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/<pseudo>/drone-detect.git"  # TODO : URL du dépôt
BASE = Path("/kaggle/working") if Path("/kaggle").exists() else Path("/content")
REPO = BASE / "drone-detect"

if not REPO.exists():
    !git clone --depth 1 {REPO_URL} {REPO}
os.chdir(REPO)
print("Dépôt :", Path.cwd())
!git log -1 --oneline

## 3. Installer les dépendances

On garde le `torch` déjà installé sur la plateforme : il correspond à son pilote CUDA, et le
réinstaller coûterait plusieurs Go. Les autres paquets sont installés aux versions de
`requirements.txt`. Les scripts gardent les réglages Ultralytics dans `.ultralytics/` du dépôt.

In [ ]:
!pip install -q ultralytics==8.4.190 lap==0.5.13 sahi==0.12.8 pycocotools==2.0.11 gdown==6.4.2 onnx==1.23.2 onnxslim==0.1.98 onnxruntime==1.31.1 scipy openpyxl

In [ ]:
import torch
import ultralytics

print("torch", torch.__version__, "| CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
print("ultralytics", ultralytics.__version__)

## 4. (Optionnel) Sauvegarder les runs sur Google Drive

Sur Colab, le disque local est effacé à la fin de la session. Avec `USE_DRIVE = True`, les runs
d'entraînement (poids compris) sont écrits sur Google Drive. Après une coupure, il suffit de
relancer les sections 1 à 5 (les données se re-téléchargent) puis la cellule de reprise de la
section 6.

In [ ]:
USE_DRIVE = False  # True sur Colab pour garder les runs après la session

PROJECT = REPO / "runs" / "train"
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/drone-detect/runs/train")
PROJECT.mkdir(parents=True, exist_ok=True)
print("Runs d'entraînement :", PROJECT)

## 5. Préparer les données

Trois sources, une seule classe `drone` (détails et licences : `docs/DATA.md`) :

- **Halmstad** (vidéos visibles, CC0-1.0, Zenodo) : drones, plus des vidéos d'oiseaux, d'avions et
  d'hélicoptères gardées comme négatifs difficiles ;
- **DUT Anti-UAV tracking** (20 séquences, Apache-2.0, Google Drive) ;
- **DUT Anti-UAV détection** (10 000 images, Apache-2.0, Google Drive), utilisé **en train
  seulement** : son découpage officiel fuit entre train et test.

Les scripts sont idempotents : une étape déjà faite n'est pas refaite.

**Si Google Drive refuse un téléchargement** (quota « too many users ») : réessayer plus tard, ou
télécharger le zip à la main (identifiants dans `scripts/data/download.py`), le déposer dans
`data/raw/<source>/` sous le même nom, puis relancer la cellule.

In [ ]:
!python scripts/data/download.py --list

In [ ]:
!python scripts/data/download.py

Conversion au format YOLO. `convert_dut_det.py` vient après `convert_dut_track.py` : il compare
chaque image de DUT détection aux séquences DUT tracking, pour écarter celles qui montrent le
décor d'une séquence de val ou de test.

In [ ]:
!python scripts/data/convert_halmstad.py
!python scripts/data/convert_dut_track.py
!python scripts/data/convert_dut_det.py

Découpage **par séquence** (jamais une même séquence dans deux splits), avec une graine fixe, puis
statistiques. Attendu : 19 495 / 1 824 / 2 326 images (train / val / test) et la ligne
`vérification : OK, aucune séquence partagée`.

In [ ]:
!python scripts/data/split.py
!python scripts/data/stats.py

## 6. Entraîner

Mêmes arguments que les runs locaux (RTX 3070 Laptop, 8 Go) :

- **R1** : YOLO11n pré-entraîné COCO, 640 px, batch 32, 3 h au plus, early stopping après 20 époques
  sans progrès ;
- **R2** : **fine-tuning de R1 à 1280 px**. Il part de `best.pt` de R1, donc R1 doit être fini. Le
  1280 px aide pour les petites cibles. Partir de R1 évite un entraînement complet à 1280, bien plus
  long. Comme les poids sont déjà spécialisés drones, le pas d'apprentissage est 5 fois plus faible
  (SGD, `lr0` 0,002), avec un échauffement court. Batch 12 (en 16, les 8 Go du GPU local ne
  suffisaient pas), 2 h au plus, patience 10.

`--time` (en heures) l'emporte sur `--epochs`, et l'early stopping (`--patience`) peut arrêter
plus tôt. Un GPU plus lent fera donc moins d'époques dans le même temps : les scores peuvent
différer de ceux du README. Avec moins de 16 Go de mémoire GPU, réduire `--batch` (ou `--batch -1`
pour AutoBatch).

Relancer une cellule d'entraînement crée un nouveau dossier (`r1_y11n_6402`…). Pour continuer un
run interrompu, utiliser la cellule de reprise.

In [ ]:
# R1 : 640 px
!python scripts/train.py --data configs/drones.yaml --model yolo11n.pt --imgsz 640 --epochs 100 --patience 20 --time 3 --batch 32 --name r1_y11n_640 --project {PROJECT}

In [ ]:
# R2 : fine-tuning à 1280 px depuis les poids de R1 (optionnel)
!python scripts/train.py --data configs/drones.yaml --model {PROJECT}/r1_y11n_640/weights/best.pt --imgsz 1280 --epochs 100 --patience 10 --time 2 --batch 12 --optimizer SGD --lr0 0.002 --close-mosaic 5 --set warmup_epochs=1 --set warmup_bias_lr=0.0 --name r2_y11n_1280_ft --project {PROJECT}

In [ ]:
# Reprise d'un run interrompu : décommenter la ligne du run concerné
# !python scripts/train.py --resume {PROJECT}/r1_y11n_640/weights/last.pt
# !python scripts/train.py --resume {PROJECT}/r2_y11n_1280_ft/weights/last.pt

## 7. Évaluer sur le test

Le test compte 2 326 images issues de séquences jamais vues à l'entraînement. `eval.py` donne
mAP50, mAP50-95, précision et rappel, le rappel par taille de cible, les scores par source et la
latence en batch 1. Sorties : `runs/eval/<run>_<imgsz>_test/`.

Référence locale (`docs/RESULTS.md`) : R1 à 640, mAP50 0,775 / mAP50-95 0,301 ; R2 à 1280,
mAP50 0,781 / mAP50-95 0,265. Toujours évaluer un modèle à son `imgsz` d'entraînement : à une
autre taille, il perd jusqu'à 26 points de mAP50 sur une partie du test.

In [ ]:
RUN = "r1_y11n_640"  # ou "r2_y11n_1280_ft"
IMGSZ = 640          # 1280 pour R2
WEIGHTS = PROJECT / RUN / "weights" / "best.pt"
EVAL_DIR = REPO / "runs" / "eval" / f"{RUN}_{IMGSZ}_test"
print(WEIGHTS, WEIGHTS.exists())

In [ ]:
!python scripts/eval.py --weights {WEIGHTS} --imgsz {IMGSZ}

In [ ]:
import json
from IPython.display import Markdown, display

display(Markdown((EVAL_DIR / "results_row.md").read_text()))
# Seuil de confiance F1-optimal (évaluateur commun), réutilisé pour la galerie d'échecs
CONF = round(json.loads((EVAL_DIR / "metrics.json").read_text())["common_eval"]["pr"]["best_f1"]["conf"], 2)
print("Seuil F1-optimal :", CONF)

**Optionnel.** Inférence par tuiles SAHI contre image entière, avec le même évaluateur. Pour gagner
du temps, `--stride 3` ne prend qu'une image de test sur trois (sous-ensemble noté dans les
sorties). Puis la galerie de cas d'échec au seuil F1-optimal, écrite dans
`outputs/failures/<run>/`.

In [ ]:
!python scripts/sahi_eval.py --weights {WEIGHTS} --imgsz {IMGSZ} --slice 640 --slice-imgsz 640 --stride 3

In [ ]:
!python scripts/failures.py --preds {EVAL_DIR}/predictions.json --conf {CONF} --out outputs/failures/{RUN} --clean

## 8. Exporter en ONNX

L'ONNX (opset 17, FP32, batch 1) est écrit à côté des poids : `best_<imgsz>.onnx`. Le script
vérifie que ses prédictions concordent avec celles de PyTorch. Un engine TensorRT n'aurait pas
d'intérêt ici : il est lié au GPU qui l'a construit. Sur un Jetson, on le reconstruit à partir de
cet ONNX (voir `docs/EXPORT.md`).

In [ ]:
!python scripts/export.py --weights {WEIGHTS} --imgsz {IMGSZ}

## 9. (Optionnel) Démo sur une séquence de test

`make_video.py` reconstruit en mp4 les 4 séquences de test réservées à la démo (`data/demo/`).
`demo.py` détecte, suit, verrouille la cible et écrit la vidéo annotée, le CSV de l'erreur de visée
et un GIF.

In [ ]:
!python scripts/data/make_video.py
!python scripts/demo.py --weights {WEIGHTS} --imgsz {IMGSZ} --source data/demo/halmstad__V_DRONE_035.mp4 --save outputs/demo/V_DRONE_035.mp4 --csv outputs/demo/V_DRONE_035.csv --gif outputs/demo/V_DRONE_035.gif

In [ ]:
from IPython.display import Image

Image(filename="outputs/demo/V_DRONE_035.gif")

## 10. Télécharger les poids et les métriques

Rassemble, pour chaque run, `best.pt`, l'ONNX, la configuration et les courbes, plus les métriques
d'évaluation, dans une archive zip. Sur Kaggle, l'archive est dans `/kaggle/working` (panneau
*Output*).

In [ ]:
import shutil

OUT = REPO / "outputs" / "colab_results"
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True)

for run_dir in sorted(PROJECT.iterdir()):
    wdir = run_dir / "weights"
    if not wdir.is_dir():
        continue
    dst = OUT / "train" / run_dir.name
    dst.mkdir(parents=True)
    for f in [*wdir.glob("best*.pt"), *wdir.glob("best*.onnx"), *wdir.glob("*_export.json")]:
        shutil.copy2(f, dst / f.name)
    for name in ["run_config.json", "run_summary.json", "args.yaml", "results.csv", "results.png"]:
        if (run_dir / name).exists():
            shutil.copy2(run_dir / name, dst / name)

for kind in ["eval", "sahi"]:
    for f in (REPO / "runs" / kind).glob("*/*.md"):
        dst = OUT / kind / f.parent.name
        dst.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dst / f.name)
        if (f.parent / "metrics.json").exists():
            shutil.copy2(f.parent / "metrics.json", dst / "metrics.json")

archive = shutil.make_archive(str(BASE / "drone-detect-results"), "zip", OUT)
print("Archive :", archive)
!cd {OUT} && find . -type f | sort

try:
    from google.colab import files

    files.download(archive)
except ImportError:
    print("Hors Colab : récupérer l'archive ci-dessus (Kaggle : panneau Output).")